In [8]:
# !pip install onnx
# !pip install onnxruntime
# !pip install xgboost

In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import joblib
import torch
import onnx
import onnxruntime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier


In [2]:
# 2. Load dataset
df = pd.read_csv("/mnt/object/train_transformed.csv")

In [3]:
df["risk_level"].value_counts()

risk_level
Low       1564054
High       234252
Medium      10228
Name: count, dtype: int64

In [4]:
# 3. Encode target column
df["risk_level"] = df["risk_level"].map({"Low": 0, "Medium": 1, "High": 2})

In [5]:
# 4. Split data
X = df.drop(columns=["risk_level"])
y = df["risk_level"]

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)


In [7]:
import mlflow
print(mlflow.get_tracking_uri())


http://129.114.25.120:8000/


In [8]:
# 5. Define MLflow experiment
mlflow.set_experiment("loan-risk-xgboost")

2025/05/10 23:52:26 INFO mlflow.tracking.fluent: Experiment with name 'loan-risk-xgboost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1746921146045, experiment_id='1', last_update_time=1746921146045, lifecycle_stage='active', name='loan-risk-xgboost', tags={}>

In [9]:
# 6. Start MLflow run
with mlflow.start_run(run_name="xgb-default-risk-classifier"):

    # 6.1 Define model and params
    params = {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 6,
        "random_state": 42,
        "objective": "multi:softprob",
        "num_class": 3,
        "use_label_encoder": False,
        "eval_metric": "mlogloss"
    }

    model = XGBClassifier(**params)

    # 6.2 Train model
    model.fit(X_train, y_train)

    # 6.3 Evaluate
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    mlflow.log_metric("val_accuracy", acc)
    print("Validation Accuracy:", acc)
    print(classification_report(y_val, y_pred, target_names=["Low", "Medium", "High"]))

    # 6.4 Log params
    mlflow.log_params(params)

    # 6.5 Save as joblib (.pth equivalent for XGBoost)
    joblib.dump(model, "loan_risk_model.pth")
    mlflow.log_artifact("loan_risk_model.pth")

    # 6.6 Export to ONNX
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType

    initial_type = [("float_input", FloatTensorType([None, X_train.shape[1]]))]
    onnx_model = convert_sklearn(model, initial_types=initial_type)
    with open("loan_risk_model.onnx", "wb") as f:
        f.write(onnx_model.SerializeToString())
    mlflow.log_artifact("loan_risk_model.onnx")

    # 6.7 Log model to MLflow tracking server
    mlflow.xgboost.log_model(model, "model")

/opt/conda/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:52:57] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation Accuracy: 0.8660822157160354
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312811
      Medium       0.00      0.00      0.00      2046
        High       0.67      0.02      0.04     46850

    accuracy                           0.87    361707
   macro avg       0.51      0.34      0.32    361707
weighted avg       0.84      0.87      0.81    361707



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🏃 View run xgb-default-risk-classifier at: http://129.114.25.120:8000/#/experiments/1/runs/c6ea0cd0c2524ebbb06c1f8047211fe9
🧪 View experiment at: http://129.114.25.120:8000/#/experiments/1


ModuleNotFoundError: No module named 'skl2onnx'